# Notebook: Clustering on MovieLens 1M
Romain Sebire - 125 009 460

**Objective:** Segment users of the MovieLens 1M dataset into meaningful groups based on their movie rating behavior, using unsupervised clustering algorithms.

**Dataset:** [MovieLens 1M](https://grouplens.org/datasets/movielens/1m/) — 1 million ratings from 6,040 users on 3,706 movies.

**Approach:** KMeans, DBSCAN, and Agglomerative clustering, with PCA and SVD for dimensionality reduction.

**Key Result:** Two distinct user profiles identified — one favoring Action/Sci-Fi/Thriller, the other preferring Comedy/Drama/Romance.

## Table of Contents

1. [Library Imports and Data Download](#Library-imports-and-data-download)
2. [Exploratory Data Analysis](#Exploratory-Data-Analysis)
3. [Data Preprocessing](#Data-Preprocessing)
4. [Applying Clustering Algorithms](#Applying-Clustering-Algorithms)
5. [Outlier Removal](#Outlier-Removal)
6. [Analysis of Cluster Number Variation](#Analysis-of-Cluster-Number-Variation)
7. [Analysis Without Dimensionality Reduction](#Analysis-Without-Dimensionality-Reduction)
8. [Dimensionality Reduction (PCA)](#Dimensionality-Reduction-(PCA))
9. [Dimensionality Reduction (SVD)](#Dimensionality-Reduction-(SVD))
10. [Conclusion](#Conclusion)

## Library imports and data download

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from scipy.spatial.distance import pdist, squareform
import os
import urllib.request
import zipfile
from collections import Counter
from sklearn.metrics import pairwise_distances
from scipy.stats import zscore

In [ ]:
# Download data
# Create a folder for the data
os.makedirs("data", exist_ok=True)

# File URLs
url_zip = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
url_readme = "https://files.grouplens.org/datasets/movielens/ml-1m-README.txt"

# Local paths
zip_path = os.path.join("data", "ml-1m.zip")
readme_path = os.path.join("data", "ml-1m.README")
extract_dir = os.path.join("data", "ml-1m")

# Download the ZIP file
if not os.path.exists(zip_path):
    print("Downloading ZIP file...")
    urllib.request.urlretrieve(url_zip, zip_path)
else:
    print("ZIP already downloaded.")

# Download the README file
if not os.path.exists(readme_path):
    print("Downloading README...")
    urllib.request.urlretrieve(url_readme, readme_path)
else:
    print("README already downloaded.")

# Extract the ZIP file
if not os.path.exists(extract_dir):
    print("Extracting ZIP...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("data")
else:
    print("Folder already extracted.")


In [ ]:
# Load data
ratings = pd.read_csv('data/ml-1m/ratings.dat', sep='::', engine='python', names=['user_id','movie_id','rating','timestamp'])
movies = pd.read_csv('data/ml-1m/movies.dat', sep='::', engine='python', encoding='latin1', names=['movie_id','title','genres'])

## Exploratory Data Analysis

In [ ]:
# Check the first rows to confirm correct import and observe the structure
print("=== Ratings ===")
print(ratings.head(5))

print("\n=== Movies ===")
print(movies.head(5))

In [ ]:
# Exploratory data analysis
print("Ratings:", len(ratings))
print("Users:", ratings['user_id'].nunique())
print("Movies:", ratings['movie_id'].nunique())
all_genres = movies['genres'].str.split('|').explode().unique()
print("Distinct genres:", len(all_genres))

In [ ]:
# Distribution of ratings
sns.countplot(data=ratings, x='rating'); 
plt.title('Rating Distribution'); 
plt.show()

In [ ]:
# Distribution of the number of movies rated per user
# Count how many movies each user rated
ratings_per_user = ratings.groupby("user_id")["movie_id"].count()

# Calculate statistics
min_ratings = ratings_per_user.min()
max_ratings = ratings_per_user.max()
mean_ratings = ratings_per_user.mean()
median_ratings = ratings_per_user.median()

# Q1 = 25th percentile (first quartile)
q1 = ratings_per_user.quantile(0.25)
# Q3 = 75th percentile (third quartile)
q3 = ratings_per_user.quantile(0.75)

print(f"First quartile (Q1): {q1}")
print(f"Third quartile (Q3): {q3}")

# Create the distribution plot
plt.figure(figsize=(10, 6))
plt.hist(ratings_per_user, bins=30, color="skyblue", edgecolor="black")
plt.title("Distribution of Number of Movies Rated per User")
plt.xlabel("Number of Movies Rated")
plt.ylabel("Number of Users")
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add vertical lines for min, max, mean and median
plt.axvline(min_ratings, color='red', linestyle='--', label=f'Mínimo: {min_ratings}')
plt.axvline(max_ratings, color='green', linestyle='--', label=f'Máximo: {max_ratings}')
plt.axvline(mean_ratings, color='orange', linestyle='-', label=f'Média: {mean_ratings:.2f}')
plt.axvline(median_ratings, color='purple', linestyle='-', label=f'Mediana: {median_ratings}')

plt.legend()
plt.show()



We observe some extreme values — users who rated up to 2,300 movies. It may be necessary to remove them, as these points will be very far from the others.

In [ ]:
# Distribution of the number of ratings per movie
# Count how many ratings each movie received
ratings_per_movie = ratings.groupby("movie_id")["user_id"].count()

# Calculate statistics
min_ratings = ratings_per_movie.min()
max_ratings = ratings_per_movie.max()
mean_ratings = ratings_per_movie.mean()
median_ratings = ratings_per_movie.median()

# Q1 = 25th percentile (first quartile)
q1 = ratings_per_movie.quantile(0.25)
# Q3 = 75th percentile (third quartile)
q3 = ratings_per_movie.quantile(0.75)

print(f"First quartile (Q1): {q1}")
print(f"Third quartile (Q3): {q3}")

# Create the distribution plot
plt.figure(figsize=(10, 6))
plt.hist(ratings_per_movie, bins=30, color="salmon", edgecolor="black")
plt.title("Distribution of Number of Ratings per Movie")
plt.xlabel("Number of Ratings per Movie")
plt.ylabel("Number of Movies")
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add vertical lines for min, max, mean and median
plt.axvline(min_ratings, color='red', linestyle='--', label=f'Mínimo: {min_ratings}')
plt.axvline(max_ratings, color='green', linestyle='--', label=f'Máximo: {max_ratings}')
plt.axvline(mean_ratings, color='orange', linestyle='-', label=f'Média: {mean_ratings:.2f}')
plt.axvline(median_ratings, color='purple', linestyle='-', label=f'Mediana: {median_ratings}')

plt.legend()
plt.show()


At least one movie received only a single rating, so it will likely need to be removed to avoid biasing the analysis.

In [ ]:
# Distribution of the number of ratings per genre
# Merge ratings with movie genres
ratings_with_genres = ratings.merge(movies[['movie_id', 'genres']], on='movie_id')

# Split genres into lists
ratings_with_genres['genre_list'] = ratings_with_genres['genres'].str.split('|')

# Explode to get one row per genre
exploded = ratings_with_genres.explode('genre_list')

# Count the number of ratings per genre
ratings_per_genre = exploded['genre_list'].value_counts()

# Calculate statistics
min_ratings = ratings_per_genre.min()
max_ratings = ratings_per_genre.max()
mean_ratings = ratings_per_genre.mean()
median_ratings = ratings_per_genre.median()
q1 = ratings_per_genre.quantile(0.25)
q3 = ratings_per_genre.quantile(0.75)

print(f"First quartile (Q1): {q1}")
print(f"Third quartile (Q3): {q3}")

# Create the distribution plot
plt.figure(figsize=(12, 6))
plt.bar(ratings_per_genre.index, ratings_per_genre.values, color="lightcoral", edgecolor="black")
plt.title("Distribution of Number of Ratings per Genre")
plt.xlabel("Genre")
plt.ylabel("Number of Ratings")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add horizontal lines for min, max, mean and median
plt.axhline(min_ratings, color='red', linestyle='--', label=f'Mínimo: {min_ratings}')
plt.axhline(max_ratings, color='green', linestyle='--', label=f'Máximo: {max_ratings}')
plt.axhline(mean_ratings, color='orange', linestyle='-', label=f'Média: {mean_ratings:.2f}')
plt.axhline(median_ratings, color='purple', linestyle='-', label=f'Mediana: {median_ratings}')

plt.legend()
plt.tight_layout()
plt.show()


Western, Film-Noir, and Documentary are the genres with the fewest ratings.

In [ ]:
# Genre distribution
genres = movies['genres'].str.get_dummies(sep='|')
genres.sum().sort_values().plot(kind='barh', figsize=(6,8)); plt.title('Genres'); plt.show()

Western, Film-Noir, and Documentary are the least represented genres in our movie list, which explains why they received fewer ratings.

In [ ]:
# Movies per year
movies['year'] = movies['title'].str.extract(r'\((\d{4})\)').astype(float)
movies['year'].value_counts().sort_index().plot(kind='bar', figsize=(12,4));
plt.title('Movies per Year')
plt.xlabel('Release Year')
plt.ylabel('Number of Movies')
plt.show()

## Data Preprocessing

In [ ]:
# Check for duplicate values
print('Duplicate ratings:', ratings.duplicated().sum(), '| Duplicate movies:', movies.duplicated().sum())

No duplicate ratings or movies.

Creation of the user-movie matrix: a matrix where each row represents a user, each column represents a movie, and the value is the rating given by the user to the movie.

Missing values are filled with 0. Replacing them with the movie's mean rating was also tested but added noise without changing the clustering results.

In [ ]:
# Create the user-movie matrix
user_movie_matrix = ratings.pivot(index='user_id', columns='movie_id', values='rating')
user_movie_matrix = user_movie_matrix.fillna(0)

We use StandardScaler to center the data at zero and reduce variance, which helps reduce distances between points uniformly. This improves the performance of distance-based clustering algorithms. We prefer StandardScaler over MinMaxScaler because it preserves the data distribution and handles extreme values better — which is important in our ratings dataset.

In [ ]:
# Standardization
scaler = StandardScaler()
user_movie_scaled = scaler.fit_transform(user_movie_matrix)

## Applying Clustering Algorithms

Application of KMeans, DBSCAN, and Agglomerative clustering algorithms with default parameters, measuring efficiency with the silhouette score, and displaying the number of elements per cluster to identify clusters with only one or two outliers.

In [ ]:
# Clustering

print("====== KMeans ======")
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans_labels = kmeans.fit_predict(user_movie_scaled)
print(f"Silhouette KMeans: {silhouette_score(user_movie_scaled, kmeans_labels):.4f}")
print("Number of elements per cluster:", np.bincount(kmeans_labels), "\n")

print("====== DBSCAN ======")
dbscan = DBSCAN(eps=3, min_samples=5)
dbscan_labels = dbscan.fit_predict(user_movie_scaled)
n_clusters_dbscan = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
print(f"Clusters DBSCAN (excluindo outliers): {n_clusters_dbscan}")
cluster_counts = Counter(dbscan_labels)
print("Number of elements per cluster:")
for label, count in sorted(cluster_counts.items()):
    print(f"  Cluster {label}: {count}")
print()

print("====== Agglomerative ======")
agglo = AgglomerativeClustering(n_clusters=5)
agglo_labels = agglo.fit_predict(user_movie_scaled)
print(f"Silhouette Agglomerative: {silhouette_score(user_movie_scaled, agglo_labels):.4f}")
print("Number of elements per cluster:", np.bincount(agglo_labels))


- KMeans achieves a very high score, but the clustering is poor: all data ends up in one cluster while the other clusters contain only outliers. Outlier removal will be necessary.
- DBSCAN fails to identify clusters — the dimensionality is too high at this point.
- Agglomerative also identified clusters with outliers.

## Outlier Removal

Given the previous results, we need to remove extreme values to prevent the formation of clusters consisting solely of these values.

DBSCAN is not used for outlier detection at this stage because the data dimensionality remains too high for the algorithm to form meaningful clusters. This prevents a reliable separation between normal points and outliers using this approach.

Instead, we adopt a three-step strategy to clean the data before clustering:

- Filtering based on the number of ratings:

    - First, we remove users with an extremely low or high number of ratings, i.e., outside the 5th–95th percentile range.

    - Then, we apply the same filter to movies, keeping only those that received a reasonable number of ratings (also between the 5th and 95th percentiles).

In [ ]:
# 1. Count how many movies each user rated
ratings_per_user = ratings.groupby("user_id")["movie_id"].count()

# 2. Count how many ratings each movie received
ratings_per_movie = ratings.groupby("movie_id")["user_id"].count()

# 3. Calculate percentiles (5% and 95%)
q05_user = ratings_per_user.quantile(0.05)
q95_user = ratings_per_user.quantile(0.95)
q05_movie = ratings_per_movie.quantile(0.05)
q95_movie = ratings_per_movie.quantile(1)

# 4. Keep only users within the defined percentiles
filtered_users = ratings_per_user[(ratings_per_user >= q05_user) & (ratings_per_user <= q95_user)]

# 5. Keep only movies within the defined percentiles
filtered_movies = ratings_per_movie[(ratings_per_movie >= q05_movie) & (ratings_per_movie <= q95_movie)]

# 6. Filter the original matrix with selected users and movies
user_movie_matrix_clean = user_movie_matrix.loc[filtered_users.index, filtered_movies.index]

- Data standardization:

    - After removing extreme users and movies, we standardize the ratings matrix so that each variable (movie) has mean 0 and standard deviation 1, which is essential for distance calculations in high-dimensional spaces.

In [ ]:
# 7. Standardize the data
scaler = StandardScaler()
user_movie_scaled_clean = scaler.fit_transform(user_movie_matrix_clean)

- Outlier detection based on distance to k nearest neighbors:
   
    - We compute the distance matrix between all users (after pre-filtering) using Euclidean distance.

    - For each user, we identify the 5 nearest neighbors (excluding themselves) and compute the mean of those distances.

    - We apply a Z-score on these means to detect which users are abnormally far from the others.

    - Users with a Z-score above 3 are considered outliers and removed from the matrix.

In [ ]:
# 8. Calculate the distance matrix between users
dist_matrix = pairwise_distances(user_movie_scaled_clean, metric='euclidean')

# 9. Calculate the mean distance to k nearest neighbors (excluding self)
k = 5
sorted_dists = np.sort(dist_matrix, axis=1)[:, 1:k+1]
mean_k_dist = sorted_dists.mean(axis=1)

# 10. Detect outliers using Z-score
z_scores_k = zscore(mean_k_dist)
outliers_knn = np.where(z_scores_k > 3)[0]
print(f"{len(outliers_knn)} outliers detected via distance to {k} nearest neighbors")

In [ ]:
# 11. Remove outliers from the filtered original matrix
user_movie_matrix_clean = user_movie_matrix_clean.drop(user_movie_matrix_clean.index[outliers_knn])

After outlier removal, it is necessary to re-standardize the data because the original distribution has been altered. This ensures that all variables maintain zero mean and unit standard deviation.

In [ ]:
# 12. Re-standardize the final data
scaler = StandardScaler()
user_movie_scaled_clean = scaler.fit_transform(user_movie_matrix_clean)

## Analysis of Cluster Number Variation

Now that outliers have been removed, let's find the optimal value of k for both algorithms, KMeans and Agglomerative.
For this, we apply each algorithm for values of k ranging from 2 to 11 and measure the effectiveness of each clustering using the silhouette score.

In [ ]:
# Analysis of cluster number variation for KMeans
scores = []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42)
    labels = km.fit_predict(user_movie_scaled_clean)
    score = silhouette_score(user_movie_scaled_clean, labels)
    scores.append(score)

plt.plot(K_range, scores, marker='o')
plt.xlabel("k")
plt.ylabel("Silhouette Score")
plt.title("Silhouette vs k (KMeans)")
plt.grid()
plt.show()

This silhouette plot for KMeans indicates that the best values of k are 2 and 3.

In [ ]:
# Analysis of cluster number variation for Agglomerative
scores = []
K_range = range(2, 11)
for k in K_range:
    ac = AgglomerativeClustering(n_clusters=k)
    labels = ac.fit_predict(user_movie_scaled_clean)
    score = silhouette_score(user_movie_scaled_clean, labels)
    scores.append(score)

plt.plot(K_range, scores, marker='o')
plt.xlabel("k")
plt.ylabel("Silhouette Score")
plt.title("Silhouette vs k (Agglomerative)")
plt.grid()
plt.show()

Same result for Agglomerative as for KMeans: the best values of k are 2 and 3.

These plots were obtained after many attempts, adjusting the outlier filtering parameters, because even with filters applied, KMeans and Agglomerative kept forming clusters with a single element. With the current parameters, we finally obtained more consistent clusters, although they are still quite imbalanced.

## Analysis Without Dimensionality Reduction

Let's apply the algorithms with the optimal values of k found.

In [ ]:
# K-Means
print("====== KMeans ======")
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans_labels = kmeans.fit_predict(user_movie_scaled_clean)
print(f"Silhouette KMeans: {silhouette_score(user_movie_scaled_clean, kmeans_labels):.4f}")
print("Number of elements per cluster:", np.bincount(kmeans_labels), "\n")

In [ ]:
# Agglomerative
print("====== Agglomerative ======")
agglo = AgglomerativeClustering(n_clusters=3)
agglo_labels = agglo.fit_predict(user_movie_scaled_clean)
print('Silhouette Agglomerative:', silhouette_score(user_movie_scaled_clean, agglo_labels))
print("Number of elements per cluster:", np.bincount(agglo_labels))

To facilitate cluster analysis, we use a function that displays two charts.
- The first chart shows the most rated movie genres by users in the cluster, indicating the most watched genres — but note that this does not mean these genres were well-rated or appreciated by users.
- The second chart is a heatmap showing the mean ratings given to each genre by users in each cluster.

In [ ]:
def analisar_clusters(user_clusters):
    """
    Analyzes the genre distribution and mean ratings per genre for each user cluster.

    Parâmetros:
    - user_clusters: DataFrame com as colunas ['user_id', 'cluster']
    """
    # Merge with ratings and movies
    ratings_with_cluster = ratings.merge(user_clusters, on='user_id')
    ratings_with_cluster = ratings_with_cluster.merge(movies, on='movie_id')

    # Extract genres as a list
    ratings_with_cluster['genre_list'] = ratings_with_cluster['genres'].apply(lambda x: x.split('|'))

    # Explode to get one row per genre
    genre_exploded = ratings_with_cluster.explode('genre_list')

    # Genre proportions per cluster
    genre_counts = genre_exploded.groupby(['cluster', 'genre_list']).size().unstack(fill_value=0)
    genre_props = genre_counts.div(genre_counts.sum(axis=1), axis=0)

    # Display bar chart of proportions
    genre_props.T.plot(kind='bar', figsize=(12,6))
    plt.title('Genre Distribution per Cluster (%)')
    plt.ylabel('Proportion')
    plt.xlabel('Genre')
    plt.xticks(rotation=45)
    plt.legend(title='Cluster')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # Mean rating per cluster and genre
    avg_ratings = genre_exploded.groupby(['cluster', 'genre_list'])['rating'].mean().unstack()

    # Display heatmap
    plt.figure(figsize=(12,6))
    sns.heatmap(avg_ratings, annot=True, fmt=".2f", cmap='coolwarm')
    plt.title('Mean Rating per Genre and Cluster')
    plt.tight_layout()
    plt.show()


In [ ]:
# Let's analyze the clusters obtained by KMeans.

user_clusters = pd.DataFrame({
    'user_id': user_movie_matrix_clean.index,
    'cluster': kmeans.labels_
})

analisar_clusters(user_clusters)

This first analysis appears to group two distinct types of users: a first type (cluster 1) that mainly watches Action, Horror, Sci-Fi, and Thriller. A second type (clusters 0 and 2) prefers genres such as Comedy, Drama, Romance, and Musical.

The heatmap, in turn, shows that cluster 1 groups people who tend to give lower ratings, regardless of genre.

In [ ]:
# Let's analyze the clusters obtained by Agglomerative.
user_clusters = pd.DataFrame({
    'user_id': user_movie_matrix_clean.index,
    'cluster': agglo.labels_
})

analisar_clusters(user_clusters)

The clusters obtained by Agglomerative appear to have a similar distribution to KMeans: one group that mainly watches Action, Adventure, Horror, Sci-Fi, and Thriller, and a second group that prefers Comedy, Drama, and Romance — although the difference here is less pronounced.

The heatmap shows that the preferences of each cluster are nearly identical, with the same categories showing high averages and the same categories showing lower averages.

### Distance Matrix Computation

In [ ]:
# Distance matrix
dist_matrix = squareform(pdist(user_movie_scaled_clean, 'euclidean'))
plt.figure(figsize=(10, 8))
sns.heatmap(dist_matrix[:100, :100], cmap='viridis')
plt.title("Distance Matrix (First 100 Users)")
plt.show()

The distance matrix does not reveal any truly relevant information or patterns.

In [ ]:
# Distribution of mean distances between users
user_avg_dist = dist_matrix.mean(axis=1)

plt.hist(user_avg_dist, bins=30)
plt.title("Distribution of Mean Distances Between Users")
plt.xlabel("Mean Distance")
plt.ylabel("Number of Users")
plt.show()

## Dimensionality Reduction (PCA)

Let's now reduce the matrix dimensionality to be able to apply DBSCAN.

In [ ]:
# PCA analysis for dimensionality reduction
pca = PCA()
pca_t = pca.fit_transform(user_movie_scaled_clean)

plt.figure(figsize=(6,4))
plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA - Cumulative Explained Variance')
plt.grid()
plt.show()

The purpose of the cumulative explained variance is to identify how many principal components are needed to explain most of the data variance. We should not exceed 50 dimensions to apply DBSCAN; here, 50 dimensions correspond to about 20% of the data variance.

In [ ]:
# Choose the number of components
pca_n = PCA(n_components=50)
pca_data = pca_n.fit_transform(user_movie_scaled_clean)

We apply DBSCAN and check how many elements are in each cluster.

In [ ]:
# DBSCAN
dbscan = DBSCAN(eps=3, min_samples=10)
dbscan_labels = dbscan.fit_predict(pca_data)
print('Clusters DBSCAN (excluindo outliers):', len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0))
cluster_counts = Counter(dbscan_labels)
print("Number of elements per cluster:")
for label, count in sorted(cluster_counts.items()):
    print(f"  Cluster {label}: {count}")
print()

Even after adjusting the parameters, DBSCAN places most elements in cluster -1 (outliers); reducing to fewer than 50 dimensions does not solve the issue.

Let's remove the outliers detected by DBSCAN and reapply KMeans and Agglomerative.

In [ ]:
# Outlier removal

# Create a mask to keep only non-outlier points
core_mask = dbscan_labels != -1

# Apply the mask to PCA data
pca_data_core = pca_data[core_mask]

# Filter the clean user-movie matrix (row = user)
user_movie_matrix_core = user_movie_matrix_clean.iloc[core_mask]

dbscan_labels_core = dbscan_labels[core_mask]

In [ ]:
# K-Means
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans_labels = kmeans.fit_predict(pca_data_core)
print('Silhouette KMeans:', silhouette_score(pca_data_core, kmeans_labels))
print('KMeans - Number of elements per cluster:', np.bincount(kmeans_labels))

In [ ]:
# Agglomerative
agglo = AgglomerativeClustering(n_clusters=3)
agglo_labels = agglo.fit_predict(pca_data_core)
print('Silhouette Agglomerative:', silhouette_score(pca_data_core, agglo_labels))
print('Agglomerative - Number of elements per cluster:', np.bincount(agglo_labels))

## Analysis With Dimensionality Reduction (PCA)

In [ ]:
# Let's analyze the clusters obtained by KMeans.

user_clusters = pd.DataFrame({
    'user_id': user_movie_matrix_core.index,
    'cluster': kmeans.labels_
})

analisar_clusters(user_clusters)

In [ ]:
# Let's analyze the clusters obtained by Agglomerative.
user_clusters = pd.DataFrame({
    'user_id': user_movie_matrix_core.index,
    'cluster': agglo.labels_
})

analisar_clusters(user_clusters)

The results obtained by KMeans and Agglomerative are again very similar, but do not seem to provide new insights; the differences between clusters are not sufficient to draw conclusions.

In [ ]:
# Let's analyze the clusters obtained by DBSCAN.
user_clusters = pd.DataFrame({
    'user_id': user_movie_matrix_core.index,
    'cluster': dbscan_labels_core
})

analisar_clusters(user_clusters)

The clusters obtained by DBSCAN are well-defined and seem to reach the same conclusions as before dimensionality reduction: a first group watches Action, Adventure, Thriller, and Sci-Fi, while the other group (clusters 0 and 1) mainly watches Comedy, Drama, Horror, and Musical.

The heatmap shows that clusters 0 and 1 seem to group people with higher ratings, while cluster 2 provides insights into preferences: people who frequently watch Action, Adventure, Thriller, and Sci-Fi enjoy those genres but give low ratings to Animation, Children's, Film-Noir, and Mystery.

## Dimensionality Reduction (SVD)

Let's now try reducing dimensions with Truncated SVD.
As with PCA, we will reduce to 50 components.

In [ ]:
# Number of components to keep
n_components = 10

# Apply TruncatedSVD
svd = TruncatedSVD(n_components=n_components, random_state=42)
svd_data = svd.fit_transform(user_movie_matrix_clean)

In [ ]:
# Cumulative explained variance by TruncatedSVD
cumulative_variance = np.cumsum(svd.explained_variance_ratio_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, n_components+1), cumulative_variance, marker='o')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Explained Variance by TruncatedSVD')
plt.grid(True)
plt.show()

10 components explain more than 25% of the variance.

In [ ]:
# Singular values of TruncatedSVD
singular_values = svd.singular_values_

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(singular_values)+1), singular_values, marker='o')
plt.title('Valores singulares do TruncatedSVD')
plt.xlabel('Componente')
plt.ylabel('Valor singular (σ)')
plt.grid(True)
plt.show()

The singular values from the SVD decomposition drop very quickly after the first few components, indicating that most of the information is concentrated in a few dimensions. Therefore, we chose to keep only 10 components, seeking a good balance between significant dimensionality reduction and preserving most of the relevant information.

In [ ]:
# DBSCAN
dbscan = DBSCAN(eps=3, min_samples=10)
dbscan_labels = dbscan.fit_predict(pca_data)
print('Clusters DBSCAN (excluindo outliers):', len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0))
cluster_counts = Counter(dbscan_labels)
print("Number of elements per cluster:")
for label, count in sorted(cluster_counts.items()):
    print(f"  Cluster {label}: {count}")
print()

In [ ]:
# Outlier removal

# Create a mask to keep only non-outlier points
core_mask = dbscan_labels != -1

# Apply the mask to PCA data
data_core_svd = svd_data[core_mask]

# Filter the clean user-movie matrix (row = user)
user_movie_matrix_core_svd = user_movie_matrix_clean.iloc[core_mask]

dbscan_labels_core_svd = dbscan_labels[core_mask]


In [ ]:
# K-Means
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans_labels = kmeans.fit_predict(data_core_svd)
print('Silhouette KMeans:', silhouette_score(data_core_svd, kmeans_labels))
print('KMeans - Number of elements per cluster:', np.bincount(kmeans_labels))

In [ ]:
# Agglomerative
agglo = AgglomerativeClustering(n_clusters=3)
agglo_labels = agglo.fit_predict(data_core_svd)
print('Silhouette Agglomerative:', silhouette_score(data_core_svd, agglo_labels))
print('Agglomerative - Number of elements per cluster:', np.bincount(agglo_labels))

## Analysis With Dimensionality Reduction (SVD)

In [ ]:
# Let's analyze the clusters obtained by KMeans.

user_clusters = pd.DataFrame({
    'user_id': user_movie_matrix_core.index,
    'cluster': kmeans.labels_
})

analisar_clusters(user_clusters)

In [ ]:
# Let's analyze the clusters obtained by Agglomerative.

user_clusters = pd.DataFrame({
    'user_id': user_movie_matrix_core.index,
    'cluster': agglo.labels_
})

analisar_clusters(user_clusters)

In [ ]:
# Let's analyze the clusters obtained by DBSCAN.

user_clusters = pd.DataFrame({
    'user_id': user_movie_matrix_core.index,
    'cluster': dbscan_labels_core
})

analisar_clusters(user_clusters)

The results obtained by SVD are identical to those obtained by PCA, even with a greater dimensionality reduction (10 dimensions).

## Conclusion

First, outlier removal: it is very difficult to obtain well-distributed clusters when applying KMeans and Agglomerative in high dimensions. Despite numerous attempts with different methods to properly remove outliers and adjusting parameters, we consistently obtained clusters with a single element, making them unanalyzable.

Second, dimensionality reduction with PCA or SVD produces quite similar results in both cases. This was observed after several attempts with very reduced or not-so-reduced dimensions. Ultimately, we kept 50 dimensions with PCA and 10 with SVD to have both analyses, but the results were still very similar.

Finally, from this exercise, we identified two user groups:

- One that mainly watches: Action, Adventure, Sci-Fi, Thriller — and dislikes Animation, Children's, Film-Noir, and Mystery.

- Another that mainly watches: Comedy, Drama, Romance, and Musical.

If we had proceeded to implement a recommendation algorithm, it could suggest movies from the genres corresponding to each of these groups.

## Key Results

| Aspect | Detail |
|--------|--------|
| **Best clustering** | KMeans and Agglomerative (k=3) after outlier removal |
| **Dimensionality reduction** | PCA (50 components) and SVD (10 components) yield similar results |
| **User group 1** | Action, Adventure, Sci-Fi, Thriller fans — lower overall ratings |
| **User group 2** | Comedy, Drama, Romance, Musical fans — higher overall ratings |
| **Key insight** | Outlier removal is critical before clustering in high-dimensional rating data |

These clusters could serve as the foundation for a collaborative filtering recommendation system, suggesting movies from each group's preferred genres.